# 06 – Figures for the presentation (M3)

This notebook makes every chart in our presentation. Each number is **recomputed from
our data and results files**, not typed in by hand, so the charts always match the
repository.

| Chart | File | Comes from |
|---|---|---|
| H1: Simpson's paradox | `figures/m3_h1_paradox.png` | `data/features.csv` (same groups as `03_experiments`, Step 3) |
| H2: attendance | `figures/m3_h2_attendance.png` | `data/features.csv` (same groups as `03_experiments`, Step 4) |
| Bus riders | `figures/m3_bus_riders.png` | `data/features.csv` (same as `03_experiments`, Step 5) |
| Models | `figures/m3_models.png` | `results/mlflow_runs.csv` (R7) |
| Probability check | `figures/m3_calibration.png` | M2 refit exactly as in `05_validation`, scored on the test set |

## Step 1 – Load the data and set the chart style

In [ ]:
# Let the notebook find the src folder (one level up)
import sys
sys.path.append("..")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.naive_bayes import GaussianNB
from src.split import load_features, get_train_test

df = load_features()
train, test = get_train_test(df)

GPA_ORDER = ["Q1 (lowest)", "Q2", "Q3", "Q4 (highest)", "Missing"]
ATT_ORDER = ["0-25%", ">25-50%", ">50-75%", ">75-100%"]
MIN_N = 50                      # same small-group rule as R5
FIG = Path("../figures"); FIG.mkdir(exist_ok=True)

# One style for every chart (same colors mean the same thing on every slide)
BLUE, ORANGE, GREEN, GRAY = "#2a78d6", "#eb6834", "#1baf7a", "#a3a29c"
INK, MUTED = "#0b0b0b", "#52514e"
plt.rcParams.update({"font.size": 13, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#ecebe7", "axes.axisbelow": True,
                     "axes.edgecolor": "#c9c8c2", "axes.labelcolor": MUTED,
                     "xtick.color": MUTED, "ytick.color": MUTED,
                     "figure.facecolor": "white", "axes.facecolor": "white"})

def save(fig, name):
    """Save a chart into figures/ and show it."""
    fig.savefig(FIG / name, dpi=200, bbox_inches="tight")
    plt.show()

print("Rows:", len(df), "| train:", len(train), "| test:", len(test))

## Step 2 – H1: overall vs. inside each group
Same comparison as R6: students first referred during term 3 are left out, students are
grouped by prior GPA and term 1 result, and groups with fewer than 50 pairs on either
side are not shown.

In [ ]:
h1 = df[~df["referred_during_t3"]]

# Overall failure rate (the "paradox" view)
overall = h1.groupby("referred_before_t3")["failed_t3"].mean() * 100

# Failure rate inside each prior GPA x term 1 group
g = (h1.groupby(["gpa_group", "failed_t1", "referred_before_t3"])["failed_t3"]
       .agg(["size", "mean"]).reset_index()
       .pivot_table(index=["gpa_group", "failed_t1"], columns="referred_before_t3",
                    values=["size", "mean"]))
g.columns = [f"{a}_{'ref' if b else 'not'}" for a, b in g.columns]
g = g.reset_index()
g = g[(g["size_ref"] >= MIN_N) & (g["size_not"] >= MIN_N)]          # R5 rule
g["gpa_group"] = pd.Categorical(g["gpa_group"], GPA_ORDER)
g = g.sort_values(["gpa_group", "failed_t1"])

short = {"Q1 (lowest)": "Q1", "Q4 (highest)": "Q4", "Missing": "No GPA"}
labels = ["Overall"] + [f"{short.get(str(a), a)} / {'failed' if b else 'passed'} t1"
                        for a, b in zip(g["gpa_group"], g["failed_t1"])]
ref = [overall[True]] + list(g["mean_ref"] * 100)
non = [overall[False]] + list(g["mean_not"] * 100)

fig, ax = plt.subplots(figsize=(11, 5.2))
x = np.arange(len(labels)); w = 0.38
ax.bar(x - w/2, ref, w, color=ORANGE, label="Referred for tutoring", zorder=3)
ax.bar(x + w/2, non, w, color=BLUE, label="Not referred", zorder=3)
for i, (a, b) in enumerate(zip(ref, non)):
    ax.text(i - w/2, a + 1, f"{a:.1f}%", ha="center", fontsize=10, color=INK)
    ax.text(i + w/2, b + 1, f"{b:.1f}%", ha="center", fontsize=10, color=INK)
ax.axvspan(-0.5, 0.5, color="#f3f2ee", zorder=0)                   # shade the overall view
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=20, ha="right")
ax.set_ylabel("Failed in term 3 (%)"); ax.set_ylim(0, 62)
ax.legend(frameon=False, loc="upper right")
ax.set_title("Overall, referred students fail more. Inside every group, they fail less.",
             loc="left", color=INK, fontsize=14)
save(fig, "m3_h1_paradox.png")

## Step 3 – H2: failure by attendance
Referred students with sessions before term 3, same attendance groups as R5. We show
the GPA groups where failure is common enough to see a pattern (Q1, Q2, no GPA).

In [ ]:
h2 = df[df["referred_before_t3"] & (df["sessions_scheduled_before_t3"] > 0)]
rates = (h2.groupby(["gpa_group", "attendance_group"], observed=True)["failed_t3"]
           .mean().unstack()[ATT_ORDER] * 100)

show = {"Q1 (lowest)": ("Q1 (lowest GPA)", ORANGE, 0),
        "Missing":     ("No prior GPA", GREEN, 1.6),
        "Q2":          ("Q2", BLUE, -1.4)}          # last number nudges the end label
fig, ax = plt.subplots(figsize=(10, 5.2))
for grp, (name, color, nudge) in show.items():
    v = rates.loc[grp].values
    ax.plot(ATT_ORDER, v, color=color, lw=2.5, marker="o", ms=8, zorder=3)
    ax.text(3.08, v[-1] + nudge, f"{name}  {v[-1]:.1f}%", va="center", color=INK, fontsize=11)
    ax.text(-0.08, v[0], f"{v[0]:.1f}%", va="center", ha="right", color=INK, fontsize=11)
ax.set_xlim(-0.45, 3.9); ax.set_ylim(0, 50)
ax.set_xlabel("Share of tutoring sessions attended before term 3")
ax.set_ylabel("Failed in term 3 (%)")
ax.set_title("Struggling students who attended more failed much less (referred students)",
             loc="left", color=INK, fontsize=14)
save(fig, "m3_h2_attendance.png")

## Step 4 – Bus riders vs. non-riders: attendance

In [ ]:
att = (h2.groupby(["gpa_group", "bus_rider"])["attendance_rate_before_t3"]
        .mean().unstack().reindex(GPA_ORDER) * 100)
names = ["Q1 GPA", "Q2 GPA", "Q3 GPA", "Q4 GPA", "No GPA"]

fig, ax = plt.subplots(figsize=(10, 4.8)); x = np.arange(len(names))
ax.bar(x - 0.2, att[0], 0.38, color=BLUE, label="Not a bus rider", zorder=3)
ax.bar(x + 0.2, att[1], 0.38, color=ORANGE, label="Bus rider", zorder=3)
for i in range(len(names)):
    ax.text(i - 0.2, att[0].iloc[i] + 1, f"{att[0].iloc[i]:.0f}%", ha="center", fontsize=10)
    ax.text(i + 0.2, att[1].iloc[i] + 1, f"{att[1].iloc[i]:.0f}%", ha="center", fontsize=10)
ax.set_xticks(x); ax.set_xticklabels(names); ax.set_ylim(0, 75)
ax.set_ylabel("Average attendance (%)"); ax.legend(frameon=False, loc="upper right", ncol=2)
gap = (att[0] - att[1])
ax.set_title(f"Bus riders attend {gap.min():.0f}-{gap.max():.0f} points less in every GPA group",
             loc="left", color=INK, fontsize=14)
save(fig, "m3_bus_riders.png")

## Step 5 – Models (R7)
Read from the exported MLflow run table. Dots with ± SD bars (an AUC axis that does not
start at 0 should not use bars).

In [ ]:
runs = pd.read_csv("../results/mlflow_runs.csv")
runs["key"] = runs["tags.mlflow.runName"].str[:2]                      # M0 ... M5
runs = runs.sort_values("key")
nice = {"M0": "M0\nbase", "M1": "M1\n+ referral", "M2": "M2\n+ prior GPA",
        "M3": "M3\n+ GPA\n+ referral", "M4": "M4\n+ attendance\n+ flag",
        "M5": "M5\n+ attendance\n(no flag)"}
auc, sd = runs["metrics.cv_auc_mean"].values, runs["metrics.cv_auc_sd"].values
best = int(np.argmax(auc))
colors = [BLUE if i == best else GRAY for i in range(len(auc))]

fig, ax = plt.subplots(figsize=(10, 4.8)); x = np.arange(len(auc))
ax.errorbar(x, auc, yerr=sd, fmt="none", ecolor=MUTED, capsize=5, zorder=2)
ax.scatter(x, auc, s=[140 if i == best else 90 for i in range(len(auc))], color=colors, zorder=3)
for i, a in enumerate(auc):
    ax.text(i, a + 0.006, f"{a:.3f}", ha="center", fontsize=11, color=INK)
ax.set_ylim(0.75, 0.90); ax.set_xticks(x); ax.set_xticklabels([nice[k] for k in runs["key"]], fontsize=11)
ax.set_ylabel("5-fold CV AUC (± SD)")
ax.set_title(f"Prior GPA is what predicts failure. {runs['key'].iloc[best]} is chosen.",
             loc="left", color=INK, fontsize=14)
save(fig, "m3_models.png")

## Step 6 – Probability check (R8)
We refit M2 **exactly as in `05_validation`** (same features, training median for missing
GPA) and compare its predicted chance with how many really failed on the test set. This
repeats the R8 result for the chart; nothing is chosen or changed based on it.

In [ ]:
FEATURES = ["failed_t1", "bus_rider", "econ_disadvantaged", "english_learner", "iep",
            "subj_English Language Arts", "subj_Mathematics", "subj_Science",
            "prior_year_gpa", "gpa_missing"]

def prepare(data):
    """Same preparation as R7/R8: subject -> 0/1 columns, True/False -> 1/0."""
    X = pd.get_dummies(data, columns=["subject_area"], prefix="subj", dtype=int)
    for col in X.select_dtypes("bool").columns:
        X[col] = X[col].astype(int)
    return X

train_X, test_X = prepare(train), prepare(test)
med = train_X["prior_year_gpa"].median()
train_X["prior_year_gpa"] = train_X["prior_year_gpa"].fillna(med)
test_X["prior_year_gpa"] = test_X["prior_year_gpa"].fillna(med)
model = GaussianNB().fit(train_X[FEATURES], train_X["failed_t3"])
p = model.predict_proba(test_X[FEATURES])[:, 1]

cal = pd.DataFrame({"predicted": p, "failed": test_X["failed_t3"]})
bins = [0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0]
names = ["0-10%", "10-30%", "30-50%", "50-70%", "70-90%", "90-100%"]
cal["group"] = pd.cut(cal["predicted"], bins=bins, labels=names, include_lowest=True)
c = cal.groupby("group", observed=True).agg(n=("failed", "size"),
                                            pred=("predicted", "mean"),
                                            actual=("failed", "mean")) * [1, 100, 100]
print(c.round(1))

fig, ax = plt.subplots(figsize=(10, 4.8)); x = np.arange(len(c))
ax.bar(x - 0.2, c["pred"], 0.38, color=GRAY, label="Model's predicted chance", zorder=3)
ax.bar(x + 0.2, c["actual"], 0.38, color=BLUE, label="Actually failed", zorder=3)
for i in range(len(c)):
    ax.text(i - 0.2, c["pred"].iloc[i] + 1.5, f"{c['pred'].iloc[i]:.0f}%", ha="center", fontsize=10)
    ax.text(i + 0.2, c["actual"].iloc[i] + 1.5, f"{c['actual'].iloc[i]:.0f}%", ha="center", fontsize=10)
ax.set_xticks(x); ax.set_xticklabels([f"{g}\n(n={int(n):,})" for g, n in zip(c.index, c["n"])], fontsize=10)
ax.set_ylim(0, 110); ax.set_ylabel("Percent"); ax.set_xlabel("Predicted risk group (test set)")
ax.legend(frameon=False, loc="upper left")
top = c.iloc[-1]
ax.set_title(f"M2 ranks well but is overconfident: '{top['pred']:.0f}%' means about {top['actual']:.0f}%",
             loc="left", color=INK, fontsize=14)
save(fig, "m3_calibration.png")

**Result:** all five charts are saved in `figures/`. Because every number is computed
here from `data/features.csv` and `results/mlflow_runs.csv`, re-running this notebook
always gives charts that match the repository.